# Teaching a GRU to write: character-level text generation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudanum/MIA5150/blob/main/notebooks/sequential-models-rnn-gru-lstm/05_char_text_generation.ipynb)

In this lab we train a small recurrent network to write like Shakespeare, one character at a time. The model never sees a word, a rule of grammar, or a dictionary: it only learns to predict the *next character* from the characters before it. Yet after a few thousand training steps it produces plausible words, speaker turns, and stage-like formatting. This is the same training objective behind every large language model, just one generation of architecture earlier.

**Learning goals**

- Recognize next-character prediction as the exact next-token objective used to train GPT-style models.
- Build a character vocabulary, encode text as integers, and form training batches of random crops whose targets are shifted by one.
- Implement an `Embedding → GRU → Linear` language model in PyTorch and count its parameters.
- Train it with cross-entropy loss and watch the samples evolve from gibberish to pseudo-Shakespeare.
- Sample with **temperature** and reason about the coherence vs. diversity trade-off.

## 1. Next-character prediction *is* language modeling

A language model estimates the probability of the next token given everything before it:

$$p_\theta(c_{t+1} \mid c_1, c_2, \ldots, c_t)$$

In this lab every "token" $c_t$ is a single **character**. If we can predict the next character well, we can generate text: predict a character, append it, feed it back in, repeat. That loop (*predict, sample, append, repeat*) is exactly how GPT writes text. The only differences are what a token is (GPT uses subword pieces; we use characters) and what computes the prediction (GPT uses a Transformer; we use a GRU).

Training needs no labels beyond the text itself. Shift the text by one character and input/target pairs make themselves: given the input `"to be or"`, the target is `"o be or "`. This is **self-supervised learning**, and it scales to all the text on the internet.

At each position the model outputs one score (a *logit*) per vocabulary character. Softmax turns the logits into a distribution, and training minimizes the cross-entropy, i.e. the negative log-probability assigned to the true next character, averaged over positions:

$$\mathcal{L} \;=\; -\,\frac{1}{T} \sum_{t=1}^{T} \log p_\theta(c_{t+1} \mid c_{\le t})$$

A model that guessed uniformly over a vocabulary of 65 characters would score $\ln 65 \approx 4.17$ nats. Anything below that is structure the model has learned.

In [ ]:
import random
import time
import urllib.request

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

# reproducibility: same weight init and same random crops every run
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# ---- Config: the knobs that control runtime, model size, and logging ----
SEQ_LEN      = 100   # length of each training crop (how much context the model sees)
BATCH_SIZE   = 64    # random crops per training step
EMB_DIM      = 128   # learned character embedding size
HIDDEN       = 256   # GRU hidden-state size, per layer
NUM_LAYERS   = 2     # stacked GRU layers
STEPS        = 2000  # training steps
LR           = 3e-3  # Adam learning rate (see the note in section 4)
GRAD_CLIP    = 1.0   # max gradient norm: RNNs can suffer exploding gradients
LOG_EVERY    = 100   # print the loss this often
SAMPLE_EVERY = 500   # generate a sample this often, to watch the model learn

## 2. Data: Tiny Shakespeare

We use **Tiny Shakespeare** (~1.1 MB), the classic `char-rnn` corpus: a concatenation of Shakespeare's plays. It is small enough to train on in minutes, but rich enough that a good model picks up character names (`KING`, `ROMEO`), colon-terminated speaker turns, and a hint of blank-verse rhythm.

The cells below download the text, build the character vocabulary, and encode the whole corpus as integers.

In [ ]:
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = urllib.request.urlopen(URL).read().decode("utf-8")

print(f"corpus size: {len(text):,} characters")
print("---- first 300 characters ----")
print(text[:300])

In [ ]:
# the vocabulary is simply the set of distinct characters in the corpus
chars = sorted(set(text))
VOCAB_SIZE = len(chars)
stoi = {c: i for i, c in enumerate(chars)}  # character -> integer index
itos = {i: c for c, i in stoi.items()}      # integer index -> character

print(f"vocabulary size: {VOCAB_SIZE}")
print(repr("".join(chars)))

# encode the entire corpus as one long tensor of indices
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)

def decode(indices):
    """Turn a tensor/list of indices back into a string."""
    return "".join(itos[int(i)] for i in indices)

print("encoded:", data[:24].tolist())
print("decoded:", repr(decode(data[:24])))

### Training batches: random crops with targets shifted by one

Each training example is a random crop of `SEQ_LEN` consecutive characters. Its target is the *same crop shifted one position to the right*, so every input position is supervised by the character that actually followed it. Random crops (rather than marching through the corpus in order) decorrelate consecutive steps and work well in practice for this exercise.

In [ ]:
def get_batch(batch_size):
    """Return x, y of shape (batch_size, SEQ_LEN): random crops and their +1 shifts."""
    starts = torch.randint(0, len(data) - SEQ_LEN - 1, (batch_size,))
    x = torch.stack([data[i : i + SEQ_LEN] for i in starts])
    y = torch.stack([data[i + 1 : i + SEQ_LEN + 1] for i in starts])
    return x.to(device), y.to(device)

x_demo, y_demo = get_batch(4)
print("x shape:", tuple(x_demo.shape), " y shape:", tuple(y_demo.shape))
print("input  row 0:", repr(decode(x_demo[0][:40])))
print("target row 0:", repr(decode(y_demo[0][:40])), "  <- same text, one character ahead")

## 3. The model: Embedding → GRU → Linear

Three stages:

1. **Embedding**, a learned $65 \times 128$ lookup table that maps each character index to a 128-dimensional vector the network can compute with. Similar characters (the vowels, the punctuation marks) end up with similar embeddings, for free, from training.
2. **GRU** (Gated Recurrent Unit): 2 stacked layers, hidden size 256. It reads the embedded sequence one character at a time and carries a hidden state $h_t$, a running summary of everything read so far. Its gates decide what to keep and what to refresh:

$$z_t = \sigma(W_z x_t + U_z h_{t-1}), \qquad \tilde{h}_t = \tanh(W_h x_t + U_h\,(r_t \odot h_{t-1})), \qquad h_t = (1 - z_t)\odot h_{t-1} + z_t \odot \tilde{h}_t$$

3. **Linear**, $256 \to 65$: scores every vocabulary character from the hidden state at each position. A softmax over these logits gives the next-character distribution.

The whole model is well under a million parameters (GPT-2 small has 124 million), yet it learns a surprising amount of English and a little Shakespeare.

In [ ]:
class CharGRU(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.gru = nn.GRU(emb_dim, hidden, num_layers=num_layers, batch_first=True)
        self.head = nn.Linear(hidden, vocab_size)

    def forward(self, x, h=None):
        # x: (batch, time) character indices; h: GRU hidden state (zeros if None)
        emb = self.embedding(x)              # (batch, time, emb_dim)
        out, h = self.gru(emb, h)            # (batch, time, hidden)
        logits = self.head(out)              # (batch, time, vocab_size)
        return logits, h

model = CharGRU(VOCAB_SIZE, EMB_DIM, HIDDEN, NUM_LAYERS).to(device)

total = sum(p.numel() for p in model.parameters())
for name, module in model.named_children():
    print(f"{name:>10}: {sum(p.numel() for p in module.parameters()):>9,} parameters")
print(f"{'total':>10}: {total:>9,} parameters")

### Sampling: turning logits into text

Generation runs the model one character at a time, always feeding back the character it just produced, so the GRU hidden state carries the story so far. Instead of always taking the argmax (which quickly falls into loops), we **sample** from the softmax distribution, scaled by a **temperature** $T$ (section 5 explores $T$ in depth). We will call this function inside the training loop to watch the model improve.

In [ ]:
@torch.no_grad()
def generate(model, seed, n=400, temperature=1.0):
    """Generate n characters continuing `seed`, sampling from softmax(logits / T)."""
    model.eval()   # good habit: eval mode disables dropout etc. during generation
    idx = torch.tensor([[stoi[c] for c in seed]], dtype=torch.long, device=device)

    # warm up: run the seed through the GRU; its final position scores what comes next
    logits, h = model(idx)
    next_logits = logits[0, -1, :]
    out = seed

    for _ in range(n):
        probs = F.softmax(next_logits / temperature, dim=-1)
        nxt = torch.multinomial(probs, num_samples=1)   # sample, don't argmax
        out += itos[nxt.item()]
        logits, h = model(nxt.view(1, 1), h)            # feed the new character back in
        next_logits = logits[0, -1, :]

    model.train()
    return out

## 4. Training

We train with Adam and cross-entropy loss. Two practical notes:

- **Learning rate.** Adam's default is $10^{-3}$; we use $3\times 10^{-3}$ so the model makes visible progress in only 2,000 steps. Because recurrent networks occasionally produce exploding gradients, we **clip the gradient norm** to 1.0 so that one bad batch cannot wreck the run. If you ever see the loss jump up and stay there, lower the learning rate.
- **Hardware.** On Colab, switch on a free GPU first: *Runtime → Change runtime type → T4 GPU*. CPU works too, it is just slower; if you are on CPU and short on time, lowering `STEPS` still shows the full gibberish → words → pseudo-Shakespeare arc.

Watch the samples printed during training: early output should be mostly gibberish with a few short words, while later output should show real words, mostly correct spacing and punctuation, and speaker-like turns.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

losses = []
t0 = time.time()
for step in range(1, STEPS + 1):
    x, y = get_batch(BATCH_SIZE)
    logits, _ = model(x)                                   # (batch, time, vocab)
    loss = criterion(logits.reshape(-1, VOCAB_SIZE), y.reshape(-1))

    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    optimizer.step()
    losses.append(loss.item())

    if step % LOG_EVERY == 0:
        recent = sum(losses[-LOG_EVERY:]) / len(losses[-LOG_EVERY:])
        print(f"step {step:>5}/{STEPS} | loss {recent:.3f} | elapsed {time.time() - t0:.0f}s")
    if step % SAMPLE_EVERY == 0:
        print(f"\n--- sample after {step} steps (temperature 0.8) ---")
        print(generate(model, seed="KING:", n=200, temperature=0.8))
        print()

print(f"training took {time.time() - t0:.0f}s for {STEPS} steps")

In [ ]:
window = max(1, min(50, len(losses) // 10))
smoothed = np.convolve(losses, np.ones(window) / window, mode="valid")

plt.figure(figsize=(9, 4))
plt.plot(losses, alpha=0.25, color="tab:blue", label="per-step loss (noisy)")
plt.plot(range(window - 1, len(losses)), smoothed, color="tab:blue",
         linewidth=2, label=f"smoothed (window {window})")
plt.axhline(np.log(VOCAB_SIZE), linestyle="--", color="gray",
            label=f"uniform-guess baseline ln({VOCAB_SIZE}) ≈ {np.log(VOCAB_SIZE):.2f}")
plt.xlabel("training step")
plt.ylabel("cross-entropy (nats)")
plt.title("Character-level GRU on Tiny Shakespeare")
plt.legend()
plt.tight_layout()
plt.show()

> **What to notice.** The loss starts near the uniform baseline and drops fast as the model learns the easy structure (most characters are lowercase, spaces are frequent, `q` is almost always followed by `u`), then declines more slowly as it absorbs words and longer-range patterns. The per-step curve stays noisy because every step sees a different random crop, so read the smoothed line. If your final samples are still gibberish, first check that the loss fell well below the dashed baseline.

## 5. Temperature: the coherence vs. diversity dial

At generation time we sample from the softmax. **Temperature** $T$ reshapes that distribution before sampling:

$$p_i \;=\; \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

- **$T < 1$ sharpens**: the model plays it safe, choosing among its favorites. Text gets more grammatical and more repetitive; low enough and it loops the same words.
- **$T = 1$** samples the distribution the model actually learned.
- **$T > 1$ flattens**: rare characters get a real chance. More surprising and inventive, but spelling and structure break down; high enough and the output dissolves into noise.

Below, the same seed phrase at four temperatures, so only the sampling changes.

In [ ]:
seed = "To be, or not to be"
assert all(c in stoi for c in seed), "seed uses a character outside the vocabulary"

torch.manual_seed(7)   # reseed so the sweep below is reproducible
for T in [0.3, 0.7, 1.0, 1.3]:
    print("=" * 72)
    print(f"temperature = {T}")
    print("=" * 72)
    print(generate(model, seed=seed, n=400, temperature=T))
    print()

> **What to notice.** Judge each output on two axes: *local grammar* (are words spelled correctly, is punctuation sane?) and *variety* (does anything unexpected happen?). $T=0.3$ should read stiff but clean, with repeated words and phrases. $T=1.3$ should mix broken spellings with genuinely inventive strings. There is no single best temperature; it depends on whether you want reliability or surprise, and modern LLM interfaces expose exactly this dial.

## 6. Bridge: from this GRU to a modern LLM

<div style="border-left: 4px solid #7c3aed; padding: 8px 16px; background-color: #f5f3ff;">
<b>You have just trained a tiny language model end to end.</b> The recipe is the modern LLM recipe with two substitutions:
<br><br>
<b>1. Characters → subword tokens.</b> Replace the 65-character alphabet with a byte-pair vocabulary of ~50k text chunks ("the", "ing", "un", ...). Same next-token objective, same cross-entropy loss, far less spelling to relearn.
<br>
<b>2. GRU → Transformer.</b> The GRU must squeeze everything it has read into one fixed-size hidden state $h_t$. Over a long passage that is a severe bottleneck: the state forgets. The Transformer replaces recurrence with <b>attention</b>, letting every position look directly at every earlier position, so nothing has to survive the squeeze through $h_t$.
<br><br>
Everything else carries over unchanged: self-supervision from shifted text, softmax next-token prediction, and temperature sampling at generation time. Attention is where this course goes next.
</div>

## Try it yourself

1. **Swap GRU for LSTM.** Replace `nn.GRU` with `nn.LSTM` (same `HIDDEN`, same `STEPS`). Two things must change: the LSTM state is a *pair* `(h, c)`, so `forward` and `generate` have to carry a tuple, and `nn.LSTM` returns `(output, (h, c))`. Whose samples read better for the same budget? Check the parameter counts: the LSTM carries about a third more parameters than the GRU. Can you hear the difference in the text?
2. **Longer context.** Set `SEQ_LEN = 200`. Training gets slower (why?). Do the samples hold speaker turns and sentence structure together for longer stretches?
3. **Train twice as long.** Set `STEPS = 4000`. Does the smoothed loss keep falling, and do the $T=1.0$ samples visibly improve? Re-plot the loss curve and compare with the shorter run.
4. **Stretch: top-k sampling.** In `generate`, before sampling, keep only the $k$ highest-probability characters: set every other logit to $-\infty$, then renormalize. Try $k \in \{5, 20\}$ at $T=1.0$. This is a sampling scheme many production LLMs use. How does it compare with pure temperature?

## Recap

- Next-character prediction **is** next-token prediction: the exact objective behind GPT, run on a character alphabet.
- Self-supervision needs no labels: the target sequence is just the input shifted by one character.
- An `Embedding → GRU → Linear` model trained with cross-entropy learns spelling, spacing, punctuation, and speaker turns in a couple of thousand steps, with well under a million parameters.
- The per-step loss is noisy; read the smoothed curve, and check that it ends well below the uniform baseline $\ln V$.
- Temperature is the coherence vs. diversity dial: low $T$ is safe and repetitive, high $T$ is inventive and chaotic.
- The GRU's fixed-size hidden state is the bottleneck. Attention removes it, and that is the next stop in the course.